# Proyecto

## Introducción

## Objetivo

## Hipótesis
El **cambio en la cartera de crédito** mexicana **no es sensible** a cambios **reputacionales** en el sector financiero. En cambio **sí es sensible** a cambios en las variables **macroeconómicas.**

## Fuentes de Información

## Unidad Muestral
**Porcentaje de desgaste o sobrevivencia** de saldo de crédito por tipo de crédito de **mes a mes.**

# Construcción de la TAD

## Librerías

In [43]:
%time
# sistema
import os

# manipulacion datos
import pandas as pd
import numpy as np
from dbfread import DBF

# graficación
import matplotlib.pyplot as plt
import plotly.express as px
import seaborn as sns

CPU times: user 4 μs, sys: 0 ns, total: 4 μs
Wall time: 7.39 μs


## Importar Datos
Aunque de manera inicial se había utilizado un función generada con Gemini, se opta por usar una función más sencilla propia. De esta manera poder visualizar de manera mas controlada el contenido de cada carpeta y poder segmentar cada origen de datos y su tratamiento previo a la unión de la tablas.

In [44]:
# ruta con archivos
ruta_bases = "/home/brauliocb126/Documents/Diplomado/Proyecto/Bases" #cpu
#ruta_bases = "/home/brauliocbd15/Documents/Diplomado/Proyecto/Bases" #laptop

### Método Gemini
Este método se escribió con ayuda de Gemini, por lo que se deben tomar consideraciones.

In [45]:
def camina_carpetas(ruta_busqueda):
    frames_ = {}

    # Recorrer de forma recursiva subcarpetas y archivos con os.walk
    for raiz, carpetas, archivos in os.walk(ruta_busqueda):
        for archivo in archivos:
            # Obtener la ruta completa del archivo
            ruta_completa = os.path.join(raiz, archivo)
            
            # Extraer el nombre sin extensión y la extensión en minúsculas
            nombre_sin_ext, extension = os.path.splitext(archivo)
            extension = extension.lower()

            try:
                # Identificar el formato por la extensión
                if extension in ['.csv', '.txt']:
                    frames_[nombre_sin_ext] = pd.read_csv(ruta_completa)
                elif extension in ['.parquet', '.pq']:
                    frames_[nombre_sin_ext] = pd.read_parquet(ruta_completa)
                elif extension in ['.xlsx', '.xls']:
                    frames_[nombre_sin_ext] = pd.read_excel(ruta_completa)
                elif extension == '.json':
                    frames_[nombre_sin_ext] = pd.read_json(ruta_completa)
                elif extension in ['.feather', '.ft']:
                    frames_[nombre_sin_ext] = pd.read_feather(ruta_completa)
                elif extension == '.dbf':
                    tabla_aux = DBF(ruta_completa)
                    frames_[nombre_sin_ext] = pd.DataFrame(iter(tabla_aux))
                else:
                    print(f"⚠️ Formato omitido: {archivo}")
                    continue

                print(f"✅ Cargado exitosamente: {os.path.relpath(ruta_completa, ruta_busqueda)}")

            except Exception as e:
                print(f"❌ Error al cargar {archivo}: {e}")

    for nombre_var, f in frames_.items():
        # Limpiamos el nombre para asegurar que sea un nombre de variable válido en Python
        nombre_limpio = nombre_var.replace(" ", "_").replace("-", "_")
        
        # Asignamos el DataFrame a una variable global con ese nombre
        globals()[nombre_limpio] = f
        print(f"Variable creada: {nombre_limpio}")

    return frames_

### Crédito Banxico

In [46]:
# información de créditos vigentes banxico
ruta_cred_banxico = ruta_bases + "/banxico_20260906"

In [47]:
frames_banxico = camina_carpetas(ruta_busqueda = ruta_cred_banxico)

✅ Cargado exitosamente: credito_personal.xlsx
✅ Cargado exitosamente: credito_hipotecario_media_residencial.xlsx
✅ Cargado exitosamente: credito_nomina.xlsx
✅ Cargado exitosamente: credito_consumo_duradero.xlsx
✅ Cargado exitosamente: credito_tarjeta_credito.xlsx
✅ Cargado exitosamente: credito_hipotecario_vivineda_social.xlsx
Variable creada: credito_personal
Variable creada: credito_hipotecario_media_residencial
Variable creada: credito_nomina
Variable creada: credito_consumo_duradero
Variable creada: credito_tarjeta_credito
Variable creada: credito_hipotecario_vivineda_social


/home/brauliocb126/Documents/Diplomado/Proyecto/pyenv_personal/lib/python3.14/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/home/brauliocb126/Documents/Diplomado/Proyecto/pyenv_personal/lib/python3.14/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/home/brauliocb126/Documents/Diplomado/Proyecto/pyenv_personal/lib/python3.14/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/home/brauliocb126/Documents/Diplomado/Proyecto/pyenv_personal/lib/python3.14/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply op

In [48]:
nom_banxico = list(frames_banxico.keys())
nom_banxico

['credito_personal',
 'credito_hipotecario_media_residencial',
 'credito_nomina',
 'credito_consumo_duradero',
 'credito_tarjeta_credito',
 'credito_hipotecario_vivineda_social']

In [49]:
for i in nom_banxico:
    n = i.split(sep="_")[1]
    columnas_credito = {'Banco de México': 'fecha',
                        'Unnamed: 1': 'saldo_vigente_'+n}
    mod =  globals()[i]
    mod.drop(index=np.arange(0,17,1), inplace=True)
    mod.rename(columns=columnas_credito, inplace=True)
    mod['fecha'] = pd.to_datetime(mod['fecha'])
    mod['saldo_vigente_'+n] = mod['saldo_vigente_'+n].astype('float64')
    display(i, mod)

'credito_personal'

,fecha,saldo_vigente_personal
17,2011-02-01,42880.6
18,2011-03-01,45666.3
19,2011-04-01,48004.1
20,2011-05-01,50076.6
21,2011-06-01,51562.1
...,...,...
198,2026-03-01,284344.0
199,2026-04-01,286438.6
200,2026-05-01,287516.5
201,2026-06-01,290675.2


'credito_hipotecario_media_residencial'

,fecha,saldo_vigente_hipotecario
17,2003-12-01,102171.2
18,2004-01-01,102803.6
19,2004-02-01,103189.7
20,2004-03-01,103895.1
21,2004-04-01,104989.5
...,...,...
284,2026-03-01,1443593.2
285,2026-04-01,1465497.0
286,2026-05-01,1453447.4
287,2026-06-01,1478693.2


'credito_nomina'

...

,fecha,saldo_vigente_consumo
17,2003-12-01,19151.4
18,2004-01-01,19975.3
19,2004-02-01,20626.8
20,2004-03-01,22400.9
21,2004-04-01,23346.9
...,...,...
284,2026-03-01,89320.7
285,2026-04-01,87534.4
286,2026-05-01,86751.1
287,2026-06-01,89672.3


'credito_tarjeta_credito'

,fecha,saldo_vigente_tarjeta
17,2003-12-01,57924.8
18,2004-01-01,57952.1
19,2004-02-01,58940.4
20,2004-03-01,61107.5
21,2004-04-01,63730.4
...,...,...
284,2026-03-01,671719.8
285,2026-04-01,681828.6
286,2026-05-01,686256.5
287,2026-06-01,693620.0


'credito_hipotecario_vivineda_social'

,fecha,saldo_vigente_hipotecario
17,2003-12-01,19234.3
18,2004-01-01,19032.1
19,2004-02-01,19308.5
20,2004-03-01,19146.5
21,2004-04-01,19001.5
...,...,...
284,2026-03-01,57798.9
285,2026-04-01,40372.8
286,2026-05-01,59937.9
287,2026-06-01,40330.8


### CNSF

In [50]:
# información de comision de seguros y fianzas
ruta_cnsf = ruta_bases + "/cnsf_20260904"

In [51]:
frames_cnsf = camina_carpetas(ruta_busqueda= ruta_cnsf)

✅ Cargado exitosamente: info_estadistica_credito_siniestros.csv
✅ Cargado exitosamente: info_estadistica_credito_vivienda_siniestros.csv
✅ Cargado exitosamente: info_sanciones_aseguradoras.csv
✅ Cargado exitosamente: info_participacion_mercado_seguros.csv
✅ Cargado exitosamente: info_estadistica_credito_emision.csv
✅ Cargado exitosamente: info_estadistica_credito_vivienda_clientes.csv
✅ Cargado exitosamente: info_seguros_estado_resultados_sector_consolidado.csv
✅ Cargado exitosamente: info_estadistica_credito_vivienda_datos_credito.csv
✅ Cargado exitosamente: info_seguros_operaciones_ramos_practicados_sector_asegurador.csv
✅ Cargado exitosamente: info_estadistica_credito_vivienda_cobranza.csv
Variable creada: info_estadistica_credito_siniestros
Variable creada: info_estadistica_credito_vivienda_siniestros
Variable creada: info_sanciones_aseguradoras
Variable creada: info_participacion_mercado_seguros
Variable creada: info_estadistica_credito_emision
Variable creada: info_estadistica_cr

In [52]:
nom_cnsf = list(frames_cnsf.keys())
nom_cnsf

['info_estadistica_credito_siniestros',
 'info_estadistica_credito_vivienda_siniestros',
 'info_sanciones_aseguradoras',
 'info_participacion_mercado_seguros',
 'info_estadistica_credito_emision',
 'info_estadistica_credito_vivienda_clientes',
 'info_seguros_estado_resultados_sector_consolidado',
 'info_estadistica_credito_vivienda_datos_credito',
 'info_seguros_operaciones_ramos_practicados_sector_asegurador',
 'info_estadistica_credito_vivienda_cobranza']

In [53]:
for i in nom_cnsf:
    display(i, globals()[i])

'info_estadistica_credito_siniestros'

,anio,plazo,tipo_insolvencia,entidad,pais,tipo_credito,giro,tipo_poliza,moneda,forma_venta,numero_siniestros,monto_siniestro,monto_pagado,monto_deducible,monto_coaseguros,gastos_ajuste,salvamentos,monto_recuperacion,recuperacion_reaseguro
0,2015,1,Mora prolongada,Puebla,No aplica (interno),Interno,Fabricacion de productos de plastico,Global,Extranjera,Agentes Persona Fisica,1,0,0,0,0,7460,0,NaN,NaN
1,2015,5,Mora prolongada,Distrito Federal,No aplica (interno),Interno,Otras industrias alimentarias,Global,Nacional,Fuerza de Venta Interna o Casa Matriz,1,-24242,0,0,0,3124,0,NaN,NaN
2,2015,7,Insolvencia de hecho,Estado de Mexico,No aplica (interno),Interno,"Comercio al por menor de combustibles, aceites...",Global,Nacional,Agentes Persona Fisica,2,0,0,0,0,157241,425880,NaN,NaN
3,2015,7,Mora prolongada,Distrito Federal,No aplica (interno),Interno,Otras industrias alimentarias,Global,Nacional,Fuerza de Venta Interna o Casa Matriz,1,-32612,0,0,0,0,0,NaN,NaN
4,2015,7,Mora prolongada,Sonora,No aplica (interno),Interno,Otro producto al Comercio al por mayor,Global,Nacional,Directo,1,0,0,0,0,522,0,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21982,2025,365,Mora prolongada,Ciudad de Mexico,No aplica (interno),Interno,"Comercio al por mayor de maquinaria, mobiliari...",Global,Extranjera,Agentes Persona Fisica,2,874419,97474,51136,9747,1514,0,0.0,874419.0
21983,2025,365,Mora prolongada,Ciudad de Mexico,No aplica (interno),Interno,"Edicion de periodicos, revistas, libros y simi...",Global,Nacional,Agentes Persona Fisica,1,0,0,0,0,52023,0,0.0,0.0
21984,2025,366,Mora prolongada,Yucatan,No aplica (interno),Mixto,Fabricacion de maquinaria y equipo para las ac...,Global,Nacional,Agentes Persona Moral,1,0,0,0,0,825,0,0.0,0.0
21985,2025,470,Mora prolongada,Puebla,No aplica (interno),Mixto,Servicios relacionados con la mineria,Global,Extranjera,Agentes Persona Fisica,1,0,43426,0,18611,0,0,0.0,0.0


'info_estadistica_credito_vivienda_siniestros'

,anio,causa_incumplimiento,evento_origen_siniestro,numero_creditos,monto_siniestro,monto_recuperado_reaseguro,monto_pagado,valor_ultimo_avaluo,valor_recuperado
0,2015,Habitante diferente al acreditado / coacreditado,Periodo establecido en la poliza desde el inic...,1,69131,0,69131,637000,0
1,2015,Divorcio o separacion o viudez,Adjudicacion al Intermediario Financiero,79,10702193,56157,5883128,40888684,0
2,2015,Negacion de pago,Dacion en Pago (cliente entrega titulo y poses...,2,362115,0,362115,2192000,0
3,2015,Habitante diferente al acreditado / coacreditado,Adjudicacion a un tercero,7,743147,0,569408,2708970,0
4,2015,Negacion de pago,Adjudicacion a un tercero,23,2722715,17727,1107110,12023463,0
...,...,...,...,...,...,...,...,...,...
487,2025,Cambio de residencia,Periodo establecido en la poliza desde el inic...,1,321336,0,321336,2442000,0
488,2025,Insolvencia,Periodo establecido en la poliza desde el inic...,23,11986072,0,10684187,89117000,0
489,2025,Otra causa de incumplimiento,Periodo establecido en la poliza desde el inic...,8,3819364,0,3819364,22796000,0
490,2025,Preferencia por el pago hacia otras deudas,Liquidación del crédito asegurado con descuento,1,481690,0,481690,4694000,0


'info_sanciones_aseguradoras'

...

,anio,clave_administrador,apoyo_financiamiento,tipo_tasa,tipo_cartera,moneda,plazo_credito,porcentaje_cobertura,frecuencia_pago,numero_creditos,monto_enganche,monto_credito,prima_emitida,prima_cedida,prima_devengada,saldo_principal
0,2015,ABC Capital,Infonavit Apoyo Infonavit,Tasa Fija,Interes Social,UDI,300,25.0,Mensual,5,205481,1302609,8296,0,8296,1207248
1,2015,ABC Capital,Infonavit Apoyo Infonavit,Tasa Fija,Interes Social,UDI,300,30.0,Mensual,4,56747,1078199,8152,0,8152,802724
2,2015,ABC Capital,Infonavit Apoyo Infonavit,Tasa Fija,Media y Residencial,UDI,240,30.0,Mensual,6,214188,3325112,24655,0,24655,2749162
3,2015,ABC Capital,Infonavit Apoyo Infonavit,Tasa Fija,Media y Residencial,UDI,300,15.0,Mensual,1,122820,477180,2148,0,2148,429228
4,2015,ABC Capital,Infonavit Cofinavit,Tasa Fija,Interes Social,UDI,240,10.0,Mensual,2,141455,394419,1260,0,1260,226307
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
84547,2025,Adamantine Servicios,Sin Apoyo,Tasa Fija,Interes Social,UDI,300,25.0,Mensual,53,1856082,12512466,58562,0,58562,7909599
84548,2025,Adamantine Servicios,Sin Apoyo,Tasa Fija,Interes Social,UDI,360,25.0,Mensual,134,4279724,29115936,315096,0,315096,36016482
84549,2025,Adamantine Servicios,Sin Apoyo,Tasa Fija,Media y Residencial,UDI,300,15.0,Mensual,1,175000,575000,4199,0,4199,613757
84550,2025,Adamantine Servicios,Sin Apoyo,Tasa Fija,Media y Residencial,UDI,300,25.0,Mensual,10,1195307,4995834,23515,0,23515,3099449


'info_seguros_operaciones_ramos_practicados_sector_asegurador'

,FECHA,INSTITUCION,VIDA,PENSIONES,ACC_ENF,SALUD,DANOS_SA,AUTOS,GARANTIA,CRED
0,31/12/2019,"Armour Secure Insurance, S.A.de C.V.",no,no,no,no,si,no,no,no
1,31/12/2019,"A.N.A. Compañía de Seguros, S.A. de C.V.",no,no,no,no,no,si,no,no
2,31/12/2019,"Agroasemex, S.A.",si,no,si,no,si,si,no,no
3,31/12/2019,"AIG Seguros México, S.A. de C.V.",no,no,si,no,si,si,no,no
4,31/12/2019,"Allianz México, S.A., Compañía de Seguros",si,no,si,no,si,si,no,no
...,...,...,...,...,...,...,...,...,...,...
714,31/12/2025,"Virginia Surety Seguros de México, S.A. de C.V.",no,no,no,no,si,si,no,no
715,31/12/2025,"Zurich Aseguradora Mexicana, S.A. de C.V.",si,no,si,no,si,si,no,no
716,31/12/2025,"Zurich Santander Seguros México, S.A.",si,no,si,no,si,si,no,no
717,31/12/2025,"Zurich Vida, Compañía de Seguros, S.A.",si,no,si,no,no,no,no,no


'info_estadistica_credito_vivienda_cobranza'

,anio,numero_mensualidades_no_pagadas,numero_meses_mora,numero_creditos,saldo_inicial_principal
0,2015,0,0,213121,110000000000
1,2015,3,1,3,902498
2,2015,6,1,2,1124063
3,2015,22,1,1,361281
4,2015,25,1,1,246900
...,...,...,...,...,...
6944,2025,24,3,3,8004278
6945,2025,48,12,14,9792752
6946,2025,20,4,1,2669850
6947,2025,44,7,1,183014


### Condusef

In [54]:
# información de comision de defensa a usuarios de entidades financieras
ruta_condusef = ruta_bases + "/condusef_20260906"

In [55]:
frames_condusef = camina_carpetas(ruta_busqueda= ruta_condusef)

✅ Cargado exitosamente: tabla_calificaciones_producto_financiero_institucion_sector.xlsx
✅ Cargado exitosamente: portal_buro_entidades_financieras.xlsx
Variable creada: tabla_calificaciones_producto_financiero_institucion_sector
Variable creada: portal_buro_entidades_financieras


In [56]:
nom_condusef = list(frames_condusef.keys())
nom_condusef

['tabla_calificaciones_producto_financiero_institucion_sector',
 'portal_buro_entidades_financieras']

In [57]:
for i in nom_condusef:
    display(i, globals()[i])

'tabla_calificaciones_producto_financiero_institucion_sector'

,CALIFICACIONES DE SUPERVISIÓN,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6
0,Periodo: Enero - junio 2024 (Base de datos del...,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Sector,Institución Financiera,Producto,RECA,EvaInformacionProducto,EvaSupervisionProducto,CalificacionPromedioProducto
3,Banca Múltiple,"Banco Nacional de México, S. A., Integrante de...",Crédito en Cuenta Corriente,0300-440-026067/06-03695-0921,NaN,10,10
4,Banca Múltiple,"Banco Nacional de México, S. A., Integrante de...",Pagaré (PRLV),0300-003-000667/27-03262-0821,NaN,9.4,9.4
...,...,...,...,...,...,...,...
627,NaN,NaN,NaN,NaN,NaN,NaN,NaN
628,NaN,NaN,NaN,NaN,NaN,NaN,NaN
629,NaN,NaN,NaN,NaN,NaN,NaN,NaN
630,SOFOM ER,NaN,NaN,NaN,NaN,NaN,NaN


'portal_buro_entidades_financieras'

,SECTOR,Id sector,Id,INSTITUCION,TOTAL DE RECLAMACIONES CONDUSEF Y REUNE,INDICE DE RECLAMACION,RECLAMACIONES CONDUSEF,PORCENTAJE DE RESOLUCION FAVORABLE,TIEMPO DE RESPUESTA EN DIAS,RECLAMACIONES REUNE,...,INDICADOR DE RENDIMIENTO SB0,INDICADOR DE RENDIMIENTO SB 55-59,INDICADOR DE RENDIMIENTO SB 60-64,INDICADOR DE RENDIMIENTO SB 65-69,INDICADOR DE RENDIMIENTO SB 70-74,INDICADOR DE RENDIMIENTO SB 75-79,INDICADOR DE RENDIMIENTO SB 80-84,INDICADOR DE RENDIMIENTO SB 85-89,INDICADOR DE RENDIMIENTO SB 90-94,INDICADOR DE RENDIMIENTO SB 1000
0,Administradoras de fondos para el retiro y emp...,4,4,"Afore XXI Banorte, S.A. de C.V.",1113,4.407149,1113,41.100917,11.015357,-,...,4.8408,4.9316,5.2095,5.3883,5.5514,5.6402,5.8976,5.8935,5.7806,5.6356
1,Administradoras de fondos para el retiro y emp...,4,7,"Profuturo AFORE, S.A. de C.V.",483,3.488297,474,29.059829,15.667368,9,...,4.3653,4.9274,5.7441,5.6807,6.0985,6.236,6.3434,6.4462,6.2298,6.1737
2,Administradoras de fondos para el retiro y emp...,4,8,"Principal Afore, S.A. de C.V., Principal Grupo...",404,4.978859,404,30.287206,20.483544,-,...,4.9139,5.192,5.3883,5.4334,5.5226,5.6856,5.3778,5.4841,5.3626,5.2237
3,Administradoras de fondos para el retiro y emp...,4,10,"Afore Sura, S.A. de C.V.",601,4.205998,587,38.602941,18.506352,14,...,5.0158,5.1819,5.7964,5.6618,5.8225,5.8677,5.9986,5.9868,5.9303,5.9397
4,Administradoras de fondos para el retiro y emp...,4,12,"Afore Inbursa, S.A. de C.V., Grupo Financiero ...",126,4.560641,110,35.849057,10.06422,16,...,5.241,5.7975,5.4941,5.6668,5.797,6.0057,6.1043,6.1812,6.2626,6.173
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3586,ENTIDADES QUE DEJARON DE SER SOFOM,No Aplica,No Aplica,"Tiempo de Creer, S.A.P.I. de C.V., SOFOM, E.N.R.",No Aplica,No Aplica,No Aplica,No Aplica,No Aplica,No Aplica,...,No Aplica,No Aplica,No Aplica,No Aplica,No Aplica,No Aplica,No Aplica,No Aplica,No Aplica,No Aplica
3587,ENTIDADES QUE DEJARON DE SER SOFOM,No Aplica,No Aplica,"Smart Bird, S.A. de C.V., SOFOM, E.N.R.",No Aplica,No Aplica,No Aplica,No Aplica,No Aplica,No Aplica,...,No Aplica,No Aplica,No Aplica,No Aplica,No Aplica,No Aplica,No Aplica,No Aplica,No Aplica,No Aplica
3588,ENTIDADES QUE DEJARON DE SER SOFOM,No Aplica,No Aplica,"Jumafri, S.A. de C.V., SOFOM, E.N.R.",No Aplica,No Aplica,No Aplica,No Aplica,No Aplica,No Aplica,...,No Aplica,No Aplica,No Aplica,No Aplica,No Aplica,No Aplica,No Aplica,No Aplica,No Aplica,No Aplica
3589,ENTIDADES QUE DEJARON DE SER SOFOM,No Aplica,No Aplica,"CT Soluciones México, S.A. de C.V., SOFOM, E.N.R.",No Aplica,No Aplica,No Aplica,No Aplica,No Aplica,No Aplica,...,No Aplica,No Aplica,No Aplica,No Aplica,No Aplica,No Aplica,No Aplica,No Aplica,No Aplica,No Aplica


### Macro Económicas

### INEGI
Se realizará año por año, es importante tomar en cuenta que a diferecnia de las demás bases de datos, la base del INEGI tine un archivo donde se explican a detalle las variables y sus respuestas, lo cual facilitará el análisis exploratorio y el proceso de importación.

#### 2024
Descripción de la base de datos:
- **TVIVINEDA:** 
    - **LAVEVIV:** código único de la vivienda que se encuesta 
**VIVIENDA**
LAVVEVIV	vivienda_key
P0_1	cuartos_dormir
P0_2	cuartos_total
P0_3	banios_full
P0_4_1	automovil
P0_4_1A	automovil_cuantos
P0_4_2	internet
P0_4_2A	modem
P1_1	personas_total
P1_2	comparten_gasto
P1_3	grupos_gasto
FOLIO	folio
VIV_SEL	vivienda_select
ENT	estado
TLOC	habitantes
REGION	region
EST_DIS	estrato_dis
UPM_DIS	unidad_muestreo
FAC_VIV	expansion_vivienda

**HOGAR**
LLAVEHOG	hogar_key
LLAVEVIV	vivienda_key
P2_8	personas_trabajo
FOLIO	folio
VIV_SEL	vivienda_select
HOGAR	hogar
EST_DIS	estrato_dis
UPM_DIS	unidad_mustr
FAC_HOG	expansion_hogar

****

In [58]:
ruta_enif_24 = ruta_bases + "/inegi_20260906/enif_2024_csv"

In [59]:
frames_enif_24 = camina_carpetas(ruta_busqueda= ruta_enif_24)

✅ Cargado exitosamente: TSDEM.csv
✅ Cargado exitosamente: TMODULO.csv
✅ Cargado exitosamente: TVIVIENDA.csv
✅ Cargado exitosamente: THOGAR.csv
Variable creada: TSDEM
Variable creada: TMODULO
Variable creada: TVIVIENDA
Variable creada: THOGAR


In [62]:
TVIVIENDA_24 = TVIVIENDA
del TVIVIENDA
TVIVIENDA_24.sample(n=10, random_state=787)

,LLAVEVIV,P0_1,P0_2,P0_3,P0_4_1,P0_4_1A,P0_4_2,P0_4_2A,P1_1,P1_2,P1_3,FOLIO,VIV_SEL,ENT,TLOC,REGION,EST_DIS,UPM_DIS,FAC_VIV
2135,213601,2,4,1,2,NaN,1,1.0,3,1,NaN,2136,1,9,1,4,48,561,4978
12269,1227004,3,5,2,1,1.0,1,1.0,4,1,NaN,12270,4,5,1,2,31,371,1862
855,85603,1,2,1,2,NaN,2,NaN,3,1,NaN,856,3,9,1,4,49,630,3778
12252,1225317,2,2,1,2,NaN,1,1.0,2,1,NaN,12253,17,6,2,3,32,390,691
4922,492303,2,4,1,1,1.0,1,1.0,4,1,NaN,4923,3,26,1,1,151,1775,2137
12324,1232501,1,2,1,1,1.0,1,1.0,2,1,NaN,12325,1,9,1,4,50,682,3886
4967,496816,2,5,2,2,NaN,1,1.0,2,1,NaN,4968,16,32,2,3,185,2141,1249
1349,135004,1,3,1,1,2.0,2,NaN,1,1,NaN,1350,4,1,1,3,4,16,1324
3876,387704,2,4,1,1,2.0,1,1.0,5,1,NaN,3877,4,26,1,1,150,1752,2611
2304,230512,2,3,1,2,NaN,2,NaN,2,1,NaN,2305,12,18,4,3,101,1219,985


In [60]:
TSDEM_24 = TSDEM
del TSDEM
TSDEM_24.sample(n=10, random_state=787)

,LLAVESDE,LLAVEVIV,LLAVEHOG,PAREN,SEXO,EDAD,NIV,GRA,FOLIO,VIV_SEL,HOGAR,N_REN,TLOC,REGION,EST_DIS,UPM_DIS,FAC_HOG
33349,1004010101,1004010,10040101,1,2,63,2.0,4.0,10040,10,1,1,3,5,67,920,1739
4150,120608104,120608,1206081,4,1,6,NaN,NaN,1206,8,1,4,3,1,13,146,496
18423,555103102,555103,5551031,5,2,28,NaN,NaN,5551,3,1,2,3,1,13,146,496
6380,187903102,187903,1879031,3,1,17,NaN,NaN,1879,3,1,2,1,3,35,416,888
2847,83612103,83612,836121,3,2,22,NaN,NaN,836,12,1,3,4,6,63,883,3042
34030,1023809102,1023809,10238091,2,2,64,NaN,NaN,10238,9,1,2,2,3,1,3,848
10480,313703107,313703,3137031,5,1,43,NaN,NaN,3137,3,1,7,1,5,97,1167,1679
10013,299702101,299702,2997021,1,2,59,3.0,3.0,2997,2,1,1,1,2,140,1656,1446
13264,397604102,397604,3976041,2,2,63,NaN,NaN,3976,4,1,2,1,3,59,868,4829
2406,70211202,70211,702112,2,2,26,NaN,NaN,702,11,2,2,4,2,136,1589,1281


In [61]:
TMODULO_24 = TMODULO
del TMODULO
TMODULO_24.sample(n=10, random_state=787)

,LLAVEMOD,LLAVEVIV,LLAVEHOG,EDAD_V,NIV,GRA,P3_1A,P3_2,P3_3,P3_4,...,FOLIO,VIV_SEL,HOGAR,N_REN,SEXO,TLOC,REGION,EST_DIS,UPM_DIS,FAC_PER
10857,1069717102,1069717,10697171,50,3,3,2,1,2,2,...,10697,17,1,2,2,4,1,43,483,7320
12040,1186904103,1186904,11869041,32,8,5,2,6,2,2,...,11869,4,1,3,1,1,4,50,719,17610
717,70405101,70405,704051,28,8,4,2,1,2,2,...,704,5,1,1,1,1,5,122,1459,1604
1050,103101101,103101,1031011,80,8,5,2,2,2,1,...,1031,1,1,1,1,1,1,146,1734,1642
1351,132904102,132904,1329041,56,3,3,2,5,2,2,...,1329,4,1,2,2,1,3,36,436,1766
7207,710814101,710814,7108141,63,0,0,2,5,2,2,...,7108,14,1,1,1,4,6,111,1380,8076
11739,1156903101,1156903,11569031,40,9,5,2,5,2,2,...,11569,3,1,1,1,1,5,98,1194,2499
8544,843101101,843101,8431011,45,3,3,2,5,2,2,...,8431,1,1,1,1,1,6,40,458,9357
4654,458901102,458901,4589011,19,6,3,2,6,2,2,...,4589,1,1,2,1,1,5,86,1071,24926
3826,377201202,377201,3772012,33,9,3,1,5,2,2,...,3772,1,2,2,2,1,1,9,84,5848


In [63]:
THOGAR_24 = THOGAR
del THOGAR
THOGAR_24.sample(n=10, random_state=787)

,LLAVEHOG,LLAVEVIV,P2_8,FOLIO,VIV_SEL,HOGAR,EST_DIS,UPM_DIS,FAC_HOG
8161,8049021,804902,1,8049,2,1,64,890,4317
7970,7859031,785903,2,7859,3,1,49,608,4537
13280,13083201,1308320,1,13083,20,1,123,1461,2454
12643,12458031,1245803,4,12458,3,1,163,1924,1839
13176,12983041,1298304,1,12983,4,1,53,794,1193
4825,4755041,475504,2,4755,4,1,133,1563,1816
3061,3010051,301005,1,3010,5,1,29,291,1723
6595,6502091,650209,2,6502,9,1,100,1210,1538
9740,9595121,959512,0,9595,12,1,111,1380,3726
262,256071,25607,1,256,7,1,47,556,1507


#### 2021

In [64]:
ruta_enif_21 = ruta_bases + "/inegi_20260906/enif_2021_csv"

In [65]:
frames_enif_21 = camina_carpetas(ruta_busqueda= ruta_enif_21)

✅ Cargado exitosamente: TSDEM.csv
✅ Cargado exitosamente: TMODULO.csv
✅ Cargado exitosamente: TVIVIENDA.csv
✅ Cargado exitosamente: THOGAR.csv
Variable creada: TSDEM
Variable creada: TMODULO
Variable creada: TVIVIENDA
Variable creada: THOGAR


In [66]:
TSDEM_21 = TSDEM
del TSDEM
TSDEM_21.sample(n=10, random_state=787)

,FOLIO,VIV_SEL,HOGAR,N_REN,P2_3,SEXO,EDAD,NIV,GRA,TLOC,REGION,EST_DIS,UPM_DIS,FAC_HOG
40335,11459,4,1,2,2,2,21,NaN,NaN,1,2,199,1746,1547
16870,4852,4,1,1,1,1,40,2.0,2.0,4,3,67,815,3612
33919,9610,2,1,1,1,2,77,4.0,3.0,1,2,173,1493,1461
3420,991,11,1,4,3,2,17,NaN,NaN,2,1,13,173,447
31723,8998,9,1,4,3,1,17,NaN,NaN,3,3,153,1392,4340
30303,8611,18,1,2,5,1,32,NaN,NaN,2,5,143,1340,3349
19628,5625,2,1,1,2,2,41,NaN,NaN,1,3,91,895,5941
14196,4149,5,1,3,3,1,7,NaN,NaN,1,1,61,719,1538
23617,6702,8,1,2,2,2,55,NaN,NaN,4,3,104,1055,3693
9016,2596,2,1,1,1,1,58,0.0,0.0,4,6,43,422,3383


In [67]:
TMODULO_21 = TMODULO
del TMODULO
TMODULO_21.sample(n=10, random_state=787)

,FOLIO,VIV_SEL,HOGAR,N_REN,P3_1_1,P3_1_2,P3_2,P3_3,P3_4,P3_5,...,P14_3_3,P14_4_3,P14_2_4,TLOC,REGION,SEXO,EDAD,EST_DIS,UPM_DIS,FAC_ELE
2890,2839,2,1,2,3,3,5,2,2,6,...,NaN,NaN,2,1,1,2,63,49,449,5564
12628,12443,3,1,1,2,3,2,2,2,1,...,NaN,NaN,2,2,5,2,56,213,1914,13350
2149,2115,2,1,1,3,3,5,2,2,1,...,NaN,NaN,2,1,3,1,25,38,357,1050
7099,7002,9,1,3,6,3,6,2,2,1,...,NaN,NaN,2,3,5,1,20,110,1105,6057
4975,4908,5,1,2,2,6,2,2,2,1,...,NaN,NaN,2,1,6,2,58,76,826,4399
10548,10383,4,1,1,6,3,5,2,2,1,...,NaN,NaN,2,3,1,1,35,175,1600,2175
5073,5005,3,1,1,2,1,5,1,1,8,...,NaN,NaN,2,2,6,1,59,72,843,3959
8673,8549,4,1,4,6,3,6,2,2,1,...,NaN,NaN,2,1,5,2,18,147,1331,18464
2534,2489,8,1,1,3,3,1,2,2,1,...,NaN,NaN,2,2,6,2,22,41,415,2889
5178,5108,4,1,1,2,3,5,2,1,1,...,1.0,3.0,2,4,6,1,49,74,849,12125


In [68]:
TVIVIENDA_21 = TVIVIENDA
del TVIVIENDA
TVIVIENDA_21.sample(n=10, random_state=787)

,FOLIO,VIV_SEL,P0_1,P0_2,P0_3,P0_4_1,P0_4_1A,P0_4_2,P0_4_2A,P1_1,P1_2,P1_3,TLOC,REGION,EST_DIS,UPM_DIS,FAC_VIV
12838,12839,15,3,3,1,2,NaN,1,1.0,5,1,NaN,2,6,221,1965,2169
10948,10949,11,2,4,1,1,2.0,1,1.0,3,1,NaN,4,1,184,1676,3685
2232,2233,3,1,2,1,1,1.0,2,NaN,2,1,NaN,1,3,40,385,685
1900,1901,7,2,5,1,2,NaN,1,1.0,5,1,NaN,2,2,26,334,2113
2492,2493,13,1,2,0,2,NaN,2,NaN,1,1,NaN,2,6,41,415,3104
4397,4398,4,3,4,1,2,NaN,1,1.0,3,1,NaN,4,1,60,767,1074
8185,8186,7,2,4,1,2,NaN,1,1.0,1,1,NaN,3,6,137,1290,2696
9910,9911,20,1,1,0,2,NaN,2,NaN,4,1,NaN,3,2,168,1537,975
3090,3091,12,1,2,1,1,1.0,2,NaN,1,1,NaN,3,1,46,499,4494
263,264,4,2,5,1,1,2.0,1,1.0,5,1,NaN,4,3,2,51,1070


In [69]:
THOGAR_21 = THOGAR
del THOGAR
THOGAR_21.sample(n=10, random_state=787)

,FOLIO,VIV_SEL,HOGAR,P2_8,EST_DIS,UPM_DIS,FAC_HOG
13174,12971,9,1,4,223,1978,1305
7436,7331,5,2,1,120,1147,1982
13355,13149,17,1,1,229,2002,854
9604,9447,16,1,2,161,1463,1290
10007,9844,1,1,1,168,1534,1035
8544,8418,4,1,1,138,1304,2661
6511,6423,2,1,1,106,1027,3179
1786,1756,3,1,1,32,300,1266
8963,8826,5,1,1,146,1354,3801
1152,1131,2,1,1,24,186,570


#### 2018

In [70]:
ruta_enif_18 = ruta_bases + "/inegi_20260906/enif_2018_dbf"

In [71]:
frames_enif_18 = camina_carpetas(ruta_busqueda= ruta_enif_18)

✅ Cargado exitosamente: tvivienda.dbf
✅ Cargado exitosamente: tmodulo2.dbf
✅ Cargado exitosamente: tsdem.dbf
✅ Cargado exitosamente: tmodulo.dbf
Variable creada: tvivienda
Variable creada: tmodulo2
Variable creada: tsdem
Variable creada: tmodulo


In [72]:
TSDEM_18 = tsdem
del tsdem
TSDEM_18.sample(n=10, random_state=787)

,UPM,VIV_SEL,HOGAR,N_REN,SEXO,EDAD,CODIGO,INF,REGION,TLOC,EST_DIS,UPM_DIS,FAC_HOG
26274,1904830,05,01,01,1,44,2,,2,1,106,01155,2664
28255,2060219,17,01,05,1,11,,,6,4,110,01234,3183
18173,1302536,02,01,02,1,09,,,5,3,067,00839,2599
39450,2803483,05,01,02,2,51,2,1,2,1,160,01666,2056
18200,1302536,10,01,01,1,64,2,,5,3,067,00839,2599
43908,3100428,05,01,02,2,43,2,1,6,1,175,01845,1509
25254,1860309,14,01,02,2,34,2,1,3,4,098,01097,1383
39506,2803940,03,01,02,2,36,2,1,2,1,159,01670,1683
2054,0207074,14,01,03,1,11,,,1,2,006,00120,6464
9134,0761310,19,01,01,1,58,1,1,6,4,036,00406,3100


In [73]:
TMODULO_18 = tmodulo
del tmodulo
TMODULO_18.sample(n=10, random_state=787)

,UPM,VIV_SEL,HOGAR,N_REN,P3_1,P3_2,P3_3,NIV,GRA,P3_5,...,P6_19_5,P6_19_6,P6_19_9,EDAD,SEXO,REGION,TLOC,EST_DIS,UPM_DIS,FAC_PER
4077,1002933,11,01,02,2,5,01,08,4,1,...,,,,34,2,1,3,049,00733,4130
791,0300649,04,01,01,1,5,00,08,5,1,...,,,,61,1,1,1,016,00156,1399
2820,0807176,01,01,01,1,1,01,06,3,1,...,,,,28,1,1,1,042,00471,5365
2223,0660019,19,01,02,2,5,00,02,6,1,...,,,,47,2,3,4,030,00376,1181
12353,3260244,14,01,02,2,5,00,03,3,5,...,,,,53,2,3,4,179,01909,3278
3428,0912503,04,01,03,3,6,00,06,3,5,...,,,,19,2,4,1,046,00591,8208
9136,2403979,12,01,02,2,5,03,02,6,1,...,,,,50,1,2,3,131,01461,1941
1756,0505478,01,01,02,2,5,00,07,3,5,...,0,0,0,36,2,2,2,027,00312,2880
9696,2560427,11,01,01,1,6,00,02,3,1,...,,,,54,1,1,4,138,01528,1898
10840,2806302,01,01,02,2,1,01,02,6,1,...,,,,51,2,2,1,159,01707,3582


In [74]:
TMODULO2_18 = tmodulo2
del tmodulo2
TMODULO2_18.sample(n=10, random_state=787)

,UPM,VIV_SEL,HOGAR,N_REN,P7_1_1,P7_1_2,P7_1_3,P7_1_4,P7_1_5,P7_1_6,...,P13_3_3,P13_4_3,P13_2_4,EDAD,SEXO,REGION,TLOC,EST_DIS,UPM_DIS,FAC_PER
4077,1002933,11,01,02,1,1,9,9,1,1,...,5,3,2,34,2,1,3,049,00733,4130
791,0300649,04,01,01,1,2,9,2,2,1,...,1,2,2,61,1,1,1,016,00156,1399
2820,0807176,01,01,01,1,1,9,1,1,1,...,,,2,28,1,1,1,042,00471,5365
2223,0660019,19,01,02,1,1,9,1,1,1,...,,,2,47,2,3,4,030,00376,1181
12353,3260244,14,01,02,1,1,9,1,1,9,...,,,2,53,2,3,4,179,01909,3278
3428,0912503,04,01,03,1,1,9,9,1,1,...,,,2,19,2,4,1,046,00591,8208
9136,2403979,12,01,02,1,1,9,9,1,1,...,,,1,50,1,2,3,131,01461,1941
1756,0505478,01,01,02,1,1,9,9,1,1,...,,,2,36,2,2,2,027,00312,2880
9696,2560427,11,01,01,1,9,9,9,1,9,...,5,2,2,54,1,1,4,138,01528,1898
10840,2806302,01,01,02,1,1,1,1,1,1,...,,,2,51,2,2,1,159,01707,3582


In [75]:
TVIVIENDA_18 = tvivienda
del tvivienda
TVIVIENDA_18.sample(n=10, random_state=787)

,UPM,VIV_SEL,P1_1,P1_2,P1_3,REGION,TLOC,EST_DIS,UPM_DIS,FAC_VIV
7645,1910623,17,04,1,,2,2,103,01209,1477
5859,1512648,01,01,1,,5,1,082,00940,11165
12269,3101871,01,02,1,,6,1,176,01865,2013
3449,0910488,05,01,1,,4,1,046,00573,2647
8410,2161628,16,03,1,,5,4,115,01290,4118
10576,2701512,04,01,1,,6,3,149,01610,1777
3786,0918597,05,01,1,,4,1,047,00648,3115
6442,1700365,03,09,1,,5,1,093,01014,1259
10782,2760882,15,02,1,,6,4,151,01622,2004
12132,3063474,07,02,1,,5,4,167,01835,6054


#### 2015

In [76]:
ruta_enif_15 = ruta_bases + "/inegi_20260906/enif_2015_dbf"

In [77]:
frames_enif_15 = camina_carpetas(ruta_busqueda= ruta_enif_15)

✅ Cargado exitosamente: tmodulo2.DBF
✅ Cargado exitosamente: tmodulo3.DBF
✅ Cargado exitosamente: tsdem.DBF
✅ Cargado exitosamente: tvivienda.DBF
✅ Cargado exitosamente: tmodulo1.DBF
Variable creada: tmodulo2
Variable creada: tmodulo3
Variable creada: tsdem
Variable creada: tvivienda
Variable creada: tmodulo1


In [78]:
TSDEM_15 = tsdem
del tsdem
TSDEM_15.sample(n=10, random_state=787)

,UPM,VIV_SEL,HOGAR,N_REN,SEXO,EDAD,TLOC,EST_DIS,UPM_DIS,FACTOR
16213,2260359,12,01,05,2,20,4,011,00587,04778
3149,0505920,10,01,03,1,27,2,005,00139,04800
21214,3060838,03,01,04,2,10,4,011,00774,04972
14574,2101279,01,01,01,1,27,1,004,00532,08662
11428,1660584,08,01,01,1,42,4,011,00425,05180
13701,1911346,08,01,02,2,45,3,009,00512,02772
3821,0601398,01,01,03,2,24,2,005,00168,04518
7246,1105670,01,01,02,1,43,2,006,00313,06629
15905,2201768,05,01,08,2,03,1,003,00577,05313
2535,0401487,20,01,02,2,22,3,008,00109,04719


In [79]:
TMODULO1_15 = tmodulo1
del tmodulo1
TMODULO1_15.sample(n=10, random_state=787)

,UPM,VIV_SEL,HOGAR,R_SEL,P3_1,P3_2,P3_3,NIV,GRA,P3_5,...,P6_3A,P6_4,P6_5,TLOC,EST_DIS,UPM_DIS,FACTOR,SEXO,EDAD,FAC_PER
3452,1909255,04,01,04,3,6,1,6,2,1,...,,1,,1,003,00505,05536,2,30,22161
1340,0800257,02,01,01,1,4,1,3,3,1,...,,1,,1,002,00189,07332,2,46,14664
4500,2460804,06,01,03,3,2,1,6,2,1,...,,2,2,4,011,00641,04406,2,35,13218
2433,1400349,01,01,01,1,5,1,3,3,1,...,,2,2,1,003,00358,05480,1,49,21937
1977,1160558,14,01,02,3,6,1,3,3,1,...,,2,2,4,011,00317,05230,1,41,10460
3217,1800719,02,01,02,2,5,1,3,3,5,...,,1,,1,003,00463,05561,2,25,11131
811,0505920,08,01,01,1,2,1,3,3,1,...,,2,2,2,005,00139,04800,2,43,14473
2109,1203647,08,01,01,1,5,1,2,3,1,...,,2,2,2,005,00333,04721,1,52,28470
4724,2560327,08,01,02,2,5,1,2,2,5,...,,1,,4,012,00663,04264,2,70,08528
1140,0762482,13,01,01,1,2,2,0,0,5,...,,2,2,4,011,00186,05084,2,55,05084


In [80]:
TMODULO2_15 = tmodulo2
del tmodulo2
TMODULO2_15.sample(n=10, random_state=787)

,UPM,VIV_SEL,HOGAR,R_SEL,P6_6_1,P6_6_2,P6_6_3,P6_6_4,P6_6_5,P6_6_6,...,P9_3_8,P9_3_9,P9_3A,TLOC,EST_DIS,UPM_DIS,FACTOR,SEXO,EDAD,FAC_PER
3452,1909255,04,01,04,,,,,,,...,,,,1,003,00505,05536,2,30,22161
1340,0800257,02,01,01,,,,,,,...,2,2,,1,002,00189,07332,2,46,14664
4500,2460804,06,01,03,1,2,2,2,2,2,...,,,,4,011,00641,04406,2,35,13218
2433,1400349,01,01,01,2,2,2,2,2,1,...,,,,1,003,00358,05480,1,49,21937
1977,1160558,14,01,02,2,2,2,2,2,1,...,,,,4,011,00317,05230,1,41,10460
3217,1800719,02,01,02,,,,,,,...,2,2,,1,003,00463,05561,2,25,11131
811,0505920,08,01,01,1,2,2,2,2,2,...,,,,2,005,00139,04800,2,43,14473
2109,1203647,08,01,01,2,1,2,2,2,2,...,,,,2,005,00333,04721,1,52,28470
4724,2560327,08,01,02,,,,,,,...,,,,4,012,00663,04264,2,70,08528
1140,0762482,13,01,01,1,2,2,2,2,2,...,,,,4,011,00186,05084,2,55,05084


In [81]:
TMODULO3_15 = tmodulo3
del tmodulo3
TMODULO3_15.sample(n=10, random_state=787)

,UPM,VIV_SEL,HOGAR,R_SEL,P9_4_1,P9_4_2,P9_4_3,P9_4_4,P9_4_5,P9_4_6,...,P12_4_6,P12_5_6,P12_5_6A,TLOC,EST_DIS,UPM_DIS,FACTOR,SEXO,EDAD,FAC_PER
3452,1909255,04,01,04,,,,,,,...,,,,1,003,00505,05536,2,30,22161
1340,0800257,02,01,01,2,2,2,1,2,2,...,,,,1,002,00189,07332,2,46,14664
4500,2460804,06,01,03,,,,,,,...,,,,4,011,00641,04406,2,35,13218
2433,1400349,01,01,01,,,,,,,...,,,,1,003,00358,05480,1,49,21937
1977,1160558,14,01,02,,,,,,,...,,,,4,011,00317,05230,1,41,10460
3217,1800719,02,01,02,1,2,2,2,2,2,...,,,,1,003,00463,05561,2,25,11131
811,0505920,08,01,01,,,,,,,...,,,,2,005,00139,04800,2,43,14473
2109,1203647,08,01,01,,,,,,,...,,,,2,005,00333,04721,1,52,28470
4724,2560327,08,01,02,,,,,,,...,,,,4,012,00663,04264,2,70,08528
1140,0762482,13,01,01,,,,,,,...,,,,4,011,00186,05084,2,55,05084


In [82]:
TVIVIENDA_15 = tvivienda
del tvivienda
TVIVIENDA_15.sample(n=10, random_state=787)

,UPM,VIV_SEL,P1_1,P1_2,P1_3,TLOC,EST_DIS,UPM_DIS,FACTOR
3523,1960419,07,05,1,,4,011,00514,02384
929,0600537,03,02,1,,1,002,00152,05548
4058,2202433,09,04,1,,2,005,00585,05043
4049,2202336,04,05,1,,2,007,00584,09786
4320,2401459,03,04,1,,1,003,00626,05487
2838,1603498,20,03,1,,2,005,00420,03821
332,0260003,03,05,1,,4,011,00062,04545
2188,1300141,03,04,1,,1,003,00341,04488
5842,3260136,16,02,1,,4,012,00824,04325
637,0401385,05,04,1,,3,009,00108,04519


#### 2012

In [83]:
ruta_enif_12 = ruta_bases + "/inegi_20260906/enif_2012_dbf"

In [84]:
frames_enif_12 = camina_carpetas(ruta_busqueda= ruta_enif_12)

✅ Cargado exitosamente: stsdem_e2.dbf
✅ Cargado exitosamente: stvivienda_e2.dbf
✅ Cargado exitosamente: stmodulo2_e2.dbf
✅ Cargado exitosamente: stmodulo1_e2.dbf
Variable creada: stsdem_e2
Variable creada: stvivienda_e2
Variable creada: stmodulo2_e2
Variable creada: stmodulo1_e2


In [85]:
TSDEM_12 = stsdem_e2
del stsdem_e2
TSDEM_12.sample(n=10, random_state=787)

,CONTROL,VIV_SEL,HOGAR,N_REN,SEXO,EDAD,CODIGO,TL,EST_DIS,UPM_DIS,FAC_VIV,FAC_PER,ESTRATO
4509,010366,04,1,02,1,12,,1,12,00023,7693,0,2
194,066010,17,1,05,2,05,,4,42,00175,2985,0,2
19110,155016,14,1,01,1,37,1,3,32,00409,4797,9594,2
17986,295013,13,1,01,1,53,2,2,22,00776,3120,0,2
20663,206037,12,1,01,2,52,1,4,41,00534,3146,3146,1
15246,010142,01,1,03,1,69,2,1,12,00010,5189,0,2
16573,320106,02,1,02,2,55,2,1,12,00829,8426,0,2
23277,285318,05,1,04,1,19,1,1,14,00734,5598,27990,4
11487,186004,11,1,03,2,74,,4,41,00480,4219,0,1
7533,266011,18,1,05,2,84,,4,41,00694,2278,0,1


In [86]:
TMODULO1_12 = stmodulo1_e2
del stmodulo1_e2
TMODULO1_12.sample(n=10, random_state=787)

,CONTROL,VIV_SEL,HOGAR,N_INF,R_SEL,TOT_PER,P3_1,P3_2,P3_3,NIV,...,P6_20_2,P6_20_3,P6_20_9,TL,EST_DIS,UPM_DIS,FAC_VIV,FAC_PER,ESTRATO,NCONDAC
3910,220008,04,1,01,01,01,1,6,2,00,...,0,0,0,1,12,00561,5921,5921,2,10
2011,110252,02,1,01,03,04,5,6,2,01,...,0,0,1,1,12,00304,7369,22107,2,50
560,035003,15,1,01,01,01,1,6,2,08,...,0,0,1,2,23,00081,7816,7816,3,10
4398,235075,12,1,01,01,02,1,4,2,02,...,0,0,1,3,32,00621,4044,8088,2,10
317,035030,15,1,02,02,04,2,5,2,08,...,0,0,0,3,32,00084,5381,10762,2,20
3922,210366,01,1,01,01,01,1,2,2,02,...,0,0,1,1,12,00545,6034,6034,2,60
5465,306038,11,1,01,01,02,1,5,2,02,...,0,0,1,4,41,00796,3723,7446,1,10
1339,070170,04,1,01,01,05,1,5,2,08,...,0,0,0,1,12,00177,6710,13420,2,10
5426,305049,18,1,02,02,02,3,6,2,08,...,0,0,1,3,31,00787,8658,8658,1,10
2618,140253,04,1,02,02,05,2,5,2,03,...,0,0,1,1,13,00366,6142,24568,3,10


In [87]:
TMODULO2_12 = stmodulo2_e2
del stmodulo2_e2
TMODULO2_12.sample(n=10, random_state=787)

,CONTROL,VIV_SEL,HOGAR,N_INF,R_SEL,P7_1,P7_2,P7_3_1,P7_4_1,P7_5_1,...,P10_12_9,P10_12_1C,P10_12_2C,P10_12_3C,TL,EST_DIS,UPM_DIS,FAC_VIV,FAC_PER,ESTRATO
3910,220103,04,1,02,02,2,1,,,,...,0,101,104,,1,13,00569,4630,9260,3
2011,230031,03,1,01,01,1,,1,1,1,...,,,,,1,14,00596,5110,10220,4
560,110563,04,1,04,01,2,3,,,,...,,,,,1,12,00310,5805,23220,2
4398,015009,19,1,01,01,2,3,,,,...,,,,,3,32,00025,5043,15129,2
317,075021,20,1,02,01,2,1,,,,...,,,,,2,22,00182,2835,5670,2
3922,225013,14,1,02,02,2,2,,,,...,0,101,,,1,22,00588,5681,11362,2
5465,255088,03,1,02,02,1,,1,2,2,...,0,102,,,1,13,00663,5052,10104,3
1339,045011,13,1,01,01,2,2,,,,...,0,400,,,3,32,00102,4028,8056,2
5426,265090,03,1,01,01,2,3,,,,...,,,,,2,22,00691,3412,6824,2
2618,150204,04,1,01,02,2,1,,,,...,,,,,1,12,00394,6214,12428,2


In [88]:
TVIVIENDA_12 = stvivienda_e2
del stvivienda_e2
TVIVIENDA_12.sample(n=10, random_state=787)

,CONTROL,VIV_SEL,P1_1,P1_2,P1_3,TL,EST_DIS,UPM_DIS,FAC_VIV,ESTRATO
5554,305097,10,05,1,01,3,32,00790,4437,2
2128,216030,18,05,1,01,4,41,00559,3725,1
3217,230366,02,02,1,01,1,12,00610,5323,2
4201,295013,10,04,1,01,2,22,00776,3120,2
598,010227,04,01,1,01,1,12,00015,5553,2
1993,146024,10,05,1,01,4,42,00385,3737,2
1747,120191,03,02,2,02,1,12,00329,5552,2
1147,085016,09,03,1,01,3,32,00202,3504,2
5555,305097,12,05,1,01,3,32,00790,4437,2
5443,315011,11,02,1,01,2,22,00821,3872,2


## Análisis Exploratorio y Limpieza

In [ ]:
nom_ = nom_banxico + nom_condusef + nom_cnsf
nom_